# extract_contour.ipynb — V1~V4 (알고리즘 인라인)

각 버전의 **알고리즘 코드를 셀 하나에 직접** 넣었습니다(패키지에서 import하지 않음).
위에서부터 순서대로 실행하세요. ★=tip, ■=root, 색 = tip→root 순서.

| 버전 | 알고리즘 |
|---|---|
| V1 | legacy: airway-facing 최장 run + min-col tip |
| V2 | upper-envelope + anterior-clip |
| V3 | normal 필터 + dorsum-run + upper-anterior tip |
| V4 | V3 tip + V2 root/body |


In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt
from skimage.measure import find_contours
plt.rcParams["figure.dpi"] = 110

REPO = Path.cwd()
assert (REPO/"retarget").exists() and (REPO/"datasets").exists(), f"repo 루트에서 실행 (현재 {REPO})"
sys.path.insert(0, str(REPO))
from modules.utils import load_mask, mask_label_2d   # 마스크 로딩(IO)만 사용

# ---- 여기만 바꾸면 됨 ----
SUBJECT = "Subject3"
FRAMES  = [1, 30, 45, 51, 60]
N_MARKERS = 25
LBL_TONGUE, LBL_AIRWAY = 4, 5

SEG = REPO/"datasets"/"GT_Segmentations"/SUBJECT
def load2d(fr):
    return mask_label_2d(load_mask(str(SEG/f"mask_{fr}.mat")))

def show_row(fn, title, **kw):
    """fn(mask, n=..) 을 FRAMES 에 대해 한 행으로 시각화."""
    fig, ax = plt.subplots(1, len(FRAMES), figsize=(2.9*len(FRAMES), 3.2))
    for a, fr in zip(np.atleast_1d(ax), FRAMES):
        m = load2d(fr); ys, xs = np.where(m == LBL_TONGUE)
        rc = fn(m, n=N_MARKERS, **kw)
        a.imshow(m, cmap="gray"); a.contour(m == LBL_AIRWAY, levels=[0.5], colors="deepskyblue", linewidths=0.6)
        if rc is not None:
            a.plot(rc[:,1], rc[:,0], "-", c="orange", lw=1.1)
            a.scatter(rc[:,1], rc[:,0], c=np.arange(len(rc)), cmap="viridis", s=14)
            a.scatter(rc[0,1], rc[0,0], marker="*", c="red", s=150, zorder=6, edgecolor="k")
            a.scatter(rc[-1,1], rc[-1,0], marker="s", c="cyan", s=55, zorder=6, edgecolor="k")
        t = f"{title} f{fr}" + (f"\nroot_col={rc[-1,1]:.0f}" if rc is not None else "")
        a.set_title(t, fontsize=9); a.axis("off")
        a.set_xlim(xs.min()-8, xs.max()+8); a.set_ylim(ys.max()+8, ys.min()-8)
    plt.tight_layout(); plt.show()
def outward_normals(rc, tongue, eps=1.5):
    """rc:(N,2) row,col open curve → 각 점의 outward(혀 밖) 단위 normal (N,2)."""
    t = np.gradient(rc, axis=0); t = t/(np.linalg.norm(t,axis=1,keepdims=True)+1e-8)
    nrm = np.stack([-t[:,1], t[:,0]], axis=1); H, W = tongue.shape
    def inside(p):
        rr = np.clip(np.rint(p[:,0]).astype(int),0,H-1); cc = np.clip(np.rint(p[:,1]).astype(int),0,W-1)
        return tongue[rr,cc]
    return np.where((~inside(rc+eps*nrm))[:,None], nrm, -nrm)

print("frames:", FRAMES)


## 공용 primitive (모든 버전이 쓰는 building block)
low-level(경계 추출·평활·리샘플·clip·envelope). 여기부터 아래로 실행.

In [ ]:
def longest_true_run_cyclic(m):
    n = len(m)
    if m.all(): return 0, n
    f2 = np.concatenate([m, m]); best_len, best = 0, (0, 0); cur = start = 0
    for i in range(2*n):
        if f2[i]:
            if cur == 0: start = i
            cur += 1
            if cur > best_len: best_len, best = cur, (start, i+1)
        else: cur = 0
    return best[0], best[1]

def clip_posterior_spur(c, drop_frac=1.0, x_reversal=True, rev_tol=0.5):
    if len(c) < 4: return c
    z = -c[:,0]; col = c[:,1]; pk = int(np.argmax(z)); rise = max(1e-6, z[pk]-z[0]); cut = len(c)
    for i in range(pk+1, len(c)):
        if x_reversal and col[i] < col[i-1]-rev_tol: cut = i; break
        if drop_frac > 0 and z[i] < z[pk]-drop_frac*rise: cut = i; break
    return c[:max(pk+2, cut)]

def clip_anterior_drop(arc, drop_frac=0.5):
    if len(arc) < 4: return arc
    z = -arc[:,0]; pk = int(np.argmax(z))
    if pk == 0: return arc
    rise = max(1e-6, z[pk]-z[:pk+1].min()); cut = 0
    for i in range(pk-1, -1, -1):
        if z[i] < z[pk]-drop_frac*rise: cut = i+1; break
    return arc[cut:]

def smooth_closed(c, win=3):
    if win <= 1 or len(c) < win: return c
    pad = win; cc = np.vstack([c[-pad:], c, c[:pad]]); k = np.ones(win)/win
    return np.column_stack([np.convolve(cc[:,0],k,"same")[pad:-pad], np.convolve(cc[:,1],k,"same")[pad:-pad]])

def resample_rowcol(e, n, smooth_win=3):
    if len(e) < 3: return None
    r_, c_ = e[:,0].astype(float), e[:,1].astype(float)
    if smooth_win > 1 and len(e) >= smooth_win:
        k = np.ones(smooth_win)/smooth_win
        rs = np.convolve(r_,k,"same"); cs = np.convolve(c_,k,"same")
        rs[0],cs[0]=r_[0],c_[0]; rs[-1],cs[-1]=r_[-1],c_[-1]; r_,c_=rs,cs
    d = np.r_[0, np.cumsum(np.hypot(np.diff(c_), np.diff(r_)))]
    if d[-1] == 0: return None
    u = np.linspace(0, d[-1], n)
    return np.column_stack([np.interp(u,d,r_), np.interp(u,d,c_)])

def facing_arc(mask, facing_thresh=2.5, smooth_win=3):
    """airway 향한 dorsal 경계 최장 run, tip(min col)->root."""
    tongue = (mask == LBL_TONGUE)
    if tongue.sum() < 10: return None
    cs = find_contours(tongue.astype(float), 0.5)
    if not cs: return None
    c = smooth_closed(max(cs, key=len), smooth_win); airway = (mask == LBL_AIRWAY)
    if airway.sum() == 0: return c
    dt = distance_transform_edt(~airway)
    rr = np.clip(c[:,0].round().astype(int), 0, mask.shape[0]-1); cc = np.clip(c[:,1].round().astype(int), 0, mask.shape[1]-1)
    facing = dt[rr,cc] <= facing_thresh
    if facing.sum() < 5: return None
    s, e = longest_true_run_cyclic(facing); arc = c[np.arange(s,e)%len(c)]
    if arc[0,1] > arc[-1,1]: arc = arc[::-1]
    return arc

def dorsal_envelope(arc):
    """열마다 최상단(min row) 점만 -> 단일값 top surface."""
    env = {}
    for r, c in arc:
        cb = int(round(c))
        if cb not in env or r < env[cb][0]: env[cb] = (r, c)
    return np.array([env[k] for k in sorted(env)], float)
print("primitives ready")


## V1 — legacy (airway-facing 최장 run + min-col tip)

In [ ]:
def precise_contour_v1(mask, n=25, facing_thresh=2.5, smooth_win=3, clip_root=True, clip_drop_frac=1.0):
    tongue = (mask == LBL_TONGUE)
    if tongue.sum() < 10: return None
    cs = find_contours(tongue.astype(float), 0.5)
    if not cs: return None
    c = max(cs, key=len); airway = (mask == LBL_AIRWAY)
    if airway.sum() > 0:
        dt = distance_transform_edt(~airway)
        rr = np.clip(c[:,0].round().astype(int),0,mask.shape[0]-1); cc = np.clip(c[:,1].round().astype(int),0,mask.shape[1]-1)
        facing = dt[rr,cc] <= facing_thresh
        if facing.sum() >= 5:
            s, e = longest_true_run_cyclic(facing); c = c[np.arange(s,e)%len(c)]
    if c[0,1] > c[-1,1]: c = c[::-1]                 # tip = min col
    if clip_root: c = clip_posterior_spur(c, drop_frac=clip_drop_frac)
    return resample_rowcol(c, n, smooth_win)

show_row(precise_contour_v1, "V1")


## V2 — upper-envelope + anterior-clip (기본)

In [ ]:
def precise_contour_v2(mask, n=25, facing_thresh=2.5, smooth_win=3, clip_root=True, clip_drop_frac=1.0, ant_drop=0.5):
    arc = facing_arc(mask, facing_thresh, smooth_win=1)
    if arc is None: return None
    e = dorsal_envelope(arc)                          # 열별 최상단 -> 루프 없음
    if ant_drop: e = clip_anterior_drop(e, ant_drop)  # 앞쪽 급강하 컷 -> tip=apex
    if clip_root: e = clip_posterior_spur(e, drop_frac=clip_drop_frac)
    return resample_rowcol(e, n, smooth_win)

show_row(precise_contour_v2, "V2")


## V3 — normal 필터 + dorsum-run + upper-anterior tip (ContourExtract.md)
airway cleanup + outward-normal 필터 + dorsum 포함 run 선택 + 위쪽-앞 tip.

In [ ]:
from skimage.morphology import remove_small_objects, binary_closing, disk
from skimage.measure import label as sklabel

def clean_airway(airway, min_size=20, closing_radius=1, keep_largest=False):
    a = remove_small_objects(np.asarray(airway).astype(bool), min_size=min_size)
    if closing_radius > 0: a = binary_closing(a, disk(closing_radius))
    if keep_largest:
        lab = sklabel(a)
        if lab.max() > 0:
            cnt = np.bincount(lab.ravel()); cnt[0] = 0; a = lab == cnt.argmax()
    return a

def outward_normals_from_contour(cont, tongue_mask, eps=1.5):
    prev_p = np.roll(cont,1,axis=0); next_p = np.roll(cont,-1,axis=0)
    t = next_p - prev_p; t = t/(np.linalg.norm(t,axis=1,keepdims=True)+1e-8)
    n1 = np.stack([-t[:,1], t[:,0]], axis=1); H, W = tongue_mask.shape
    def inside(p):
        rr = np.clip(np.rint(p[:,0]).astype(int),0,H-1); cc = np.clip(np.rint(p[:,1]).astype(int),0,W-1)
        return tongue_mask[rr,cc]
    return np.where((~inside(cont+eps*n1))[:,None], n1, -n1)

def close_small_false_gaps_cyclic(keep, max_gap=4):
    n = len(keep); k = np.asarray(keep).copy()
    if k.all() or not k.any(): return k
    start = int(np.where(k)[0][0]); rolled = np.roll(k,-start); i = 0
    while i < n:
        if not rolled[i]:
            j = i
            while j < n and not rolled[j]: j += 1
            if (j-i) <= max_gap: rolled[i:j] = True
            i = j
        else: i += 1
    return np.roll(rolled, start)

def true_runs_cyclic_indices(keep):
    n = len(keep); keep = np.asarray(keep)
    if not keep.any(): return []
    if keep.all(): return [np.arange(n)]
    start = int(np.where(~keep)[0][0]); rolled = np.roll(keep,-start); runs = []; i = 0
    while i < n:
        if rolled[i]:
            j = i
            while j < n and rolled[j]: j += 1
            runs.append(np.array([(start+t)%n for t in range(i,j)])); i = j
        else: i += 1
    return runs

def cyclic_index_distance_to_run(idx, run, n):
    d = np.abs(np.asarray(run)-idx); return float(np.minimum(d, n-d).min())

def score_run(run, cont, normal, dorsum_idx, prev_tip=None, w_dorsum=2.5, w_length=1.0, w_ventral=1.5, w_temporal=1.0):
    n = len(cont); pts = cont[run]; nn = normal[run]
    length_score = len(run)/n
    dd = cyclic_index_distance_to_run(dorsum_idx, run, n); dorsum_score = 1.0-min(dd/max(n*0.15,1),1.0)
    ventral_score = 1.0-np.mean(nn[:,0] > 0.55)
    cmin = pts[:,1].min(); cand = np.where(pts[:,1] <= cmin+8)[0]
    tip = pts[cand[np.argmin(pts[cand,0])]] if len(cand) else pts[np.argmin(pts[:,1])]
    temporal = -min(np.linalg.norm(tip-prev_tip)/20.0, 2.0) if prev_tip is not None else 0.0
    return w_dorsum*dorsum_score + w_length*length_score + w_ventral*ventral_score + w_temporal*temporal

def choose_upper_anterior_tip(seg, tip_band_px=8):
    cols = seg[:,1]; rows = seg[:,0]; cmin = cols.min(); cand = np.where(cols <= cmin+tip_band_px)[0]
    return int(np.argmin(cols)) if len(cand)==0 else int(cand[np.argmin(rows[cand])])

def keep_dorsal_side_after_tip(seg, tip_idx):
    a = seg[tip_idx:]; b = seg[:tip_idx+1][::-1]
    def sc(s):
        if len(s) < 3: return -1e9
        return 2.0*(s[-1,1]-s[0,1]) + 0.2*len(s) + 0.05*(-np.mean(s[:,0]))
    return a if sc(a) >= sc(b) else b

def precise_contour_v3(mask, n=25, facing_thresh=2.5, normal_row_max=0.35, tip_band_px=8,
                       gap_close_pts=4, prev_tip=None, clip_root=True, clip_drop_frac=1.0,
                       airway_min_size=20, airway_closing_radius=1, airway_keep_largest=False, smooth_win=3):
    tongue = (mask == LBL_TONGUE)
    if tongue.sum() < 10: return None
    airway = clean_airway(mask == LBL_AIRWAY, airway_min_size, airway_closing_radius, airway_keep_largest)
    cs = find_contours(tongue.astype(float), 0.5)
    if not cs: return None
    cont = max(cs, key=len)
    dt = distance_transform_edt(~airway) if airway.sum() else np.full(mask.shape, 1e9)
    rr = np.clip(np.rint(cont[:,0]).astype(int),0,mask.shape[0]-1); cc = np.clip(np.rint(cont[:,1]).astype(int),0,mask.shape[1]-1)
    airway_ok = dt[rr,cc] <= facing_thresh
    normal = outward_normals_from_contour(cont, tongue)
    keep = close_small_false_gaps_cyclic(airway_ok & (normal[:,0] < normal_row_max), gap_close_pts)
    runs = true_runs_cyclic_indices(keep)
    if len(runs) == 0:                                   # fallback -> V2
        return precise_contour_v2(mask, n=n, clip_root=clip_root, clip_drop_frac=clip_drop_frac)
    dorsum_idx = int(np.argmin(cont[:,0]))
    _, best = max(((score_run(r, cont, normal, dorsum_idx, prev_tip), r) for r in runs), key=lambda x: x[0])
    seg = cont[best]
    seg = keep_dorsal_side_after_tip(seg, choose_upper_anterior_tip(seg, tip_band_px))
    if seg[-1,1] < seg[0,1]: seg = seg[::-1]
    if clip_root: seg = clip_posterior_spur(seg, drop_frac=clip_drop_frac)
    return resample_rowcol(seg, n, smooth_win)

show_row(precise_contour_v3, "V3")


## V4 — V3 tip + V2 root/body
※ 앞의 **V2·V3 셀을 먼저 실행**해야 함(그 함수를 사용).

In [ ]:
def precise_contour_v4(mask, n=25, facing_thresh=2.5, clip_root=True, clip_drop_frac=1.0, smooth_win=3, ant_drop=0.5):
    arc = facing_arc(mask, facing_thresh, smooth_win=1)
    if arc is None:
        return precise_contour_v2(mask, n=n, clip_root=clip_root, clip_drop_frac=clip_drop_frac)
    env = dorsal_envelope(arc)
    e2 = clip_anterior_drop(env, ant_drop) if ant_drop else env
    if clip_root: e2 = clip_posterior_spur(e2, drop_frac=clip_drop_frac)
    if len(e2) < 3:
        return precise_contour_v2(mask, n=n, clip_root=clip_root, clip_drop_frac=clip_drop_frac)
    root_v2 = e2[-1]                                     # V2 root anchor
    v3 = precise_contour_v3(mask, n=n, facing_thresh=facing_thresh, clip_root=clip_root,
                            clip_drop_frac=clip_drop_frac, smooth_win=smooth_win)
    tt = v3[0] if v3 is not None else e2[0]              # V3 tip anchor
    i_tt = int(np.argmin(np.hypot(env[:,0]-tt[0], env[:,1]-tt[1])))
    i_root = int(np.argmin(np.hypot(env[:,0]-root_v2[0], env[:,1]-root_v2[1])))
    lo, hi = sorted([i_tt, i_root]); seg = env[lo:hi+1].copy()
    if np.hypot(*(seg[0]-tt)) > np.hypot(*(seg[-1]-tt)): seg = seg[::-1]
    seg[0] = tt; seg[-1] = root_v2
    return resample_rowcol(seg, n, smooth_win)

show_row(precise_contour_v4, "V4")


## V5 — V1 root + tip fix
V1의 root(경계 추종, 더 posterior)를 그대로 두고 tip의 U자 감김만 제거.
※ 앞의 **공용 primitive + V3 셀**(choose_upper_anterior_tip)을 먼저 실행.

In [ ]:
def precise_contour_v5(mask, n=25, facing_thresh=2.5, smooth_win=3, clip_root=True, clip_drop_frac=1.0, tip_band_px=8):
    tongue = (mask == LBL_TONGUE)
    if tongue.sum() < 10: return None
    cs = find_contours(tongue.astype(float), 0.5)
    if not cs: return None
    c = max(cs, key=len); airway = (mask == LBL_AIRWAY)
    if airway.sum() > 0:
        dt = distance_transform_edt(~airway)
        rr = np.clip(c[:,0].round().astype(int),0,mask.shape[0]-1); cc = np.clip(c[:,1].round().astype(int),0,mask.shape[1]-1)
        facing = dt[rr,cc] <= facing_thresh
        if facing.sum() >= 5:
            s, e = longest_true_run_cyclic(facing); c = c[np.arange(s,e)%len(c)]
    if c[0,1] > c[-1,1]: c = c[::-1]
    if clip_root: c = clip_posterior_spur(c, drop_frac=clip_drop_frac)   # V1 root 확정 (그대로)
    tip_idx = choose_upper_anterior_tip(c, tip_band_px)                  # 앞쪽 apex
    if tip_idx < len(c)-3: c = c[tip_idx:]                               # wrap 제거, root 보존
    return resample_rowcol(c, n, smooth_win)

show_row(precise_contour_v5, "V5")


---
맨 위 `SUBJECT`/`FRAMES`/`N_MARKERS`만 바꿔 재실행하면 됩니다. 각 버전 셀의 함수 코드를 직접
수정해 파라미터·로직을 실험할 수 있어요. 알고리즘 설명은 `versionmanagement.md`.

## V6 — V5 + normal로 tip을 진짜 혀끝까지 연장
V5는 apex에서 멈춰 혀끝 앞면을 놓침. outward normal이 아래(underside)로 꺾이기 직전까지 tip 연장.
※ 공용 primitive + outward_normals(설정 셀) 사용. root는 V5와 동일.

In [ ]:
def precise_contour_v6(mask, n=25, facing_thresh=2.5, smooth_win=3, clip_root=True, clip_drop_frac=1.0, down_thresh=0.35):
    tongue = (mask == LBL_TONGUE)
    if tongue.sum() < 10: return None
    cs = find_contours(tongue.astype(float), 0.5)
    if not cs: return None
    c = max(cs, key=len); airway = (mask == LBL_AIRWAY)
    if airway.sum() > 0:
        dt = distance_transform_edt(~airway)
        rr = np.clip(c[:,0].round().astype(int),0,mask.shape[0]-1); cc = np.clip(c[:,1].round().astype(int),0,mask.shape[1]-1)
        facing = dt[rr,cc] <= facing_thresh
        if facing.sum() >= 5:
            s, e = longest_true_run_cyclic(facing); c = c[np.arange(s,e)%len(c)]
    if c[0,1] > c[-1,1]: c = c[::-1]
    if clip_root: c = clip_posterior_spur(c, drop_frac=clip_drop_frac)   # root = V1/V5
    nv = outward_normals(c, tongue); nrow = nv[:,0]                       # outward normal
    if smooth_win > 1 and len(nrow) >= smooth_win:
        nrow = np.convolve(nrow, np.ones(smooth_win)/smooth_win, "same")
    pk = int(np.argmin(c[:,0]))                                          # dorsum peak
    tip_idx = 0
    for i in range(pk, -1, -1):                                          # dorsum -> anterior
        if nrow[i] > down_thresh: tip_idx = i+1; break                   # underside(아래) 직전까지
    tip_idx = min(tip_idx, pk); c = c[tip_idx:]
    return resample_rowcol(c, n, smooth_win)

show_row(precise_contour_v6, "V6")


## V7 — normal 각도 급변(corner) 기반 tip (실험형)
V6와 유사하되 tip을 normal 각도가 급변하는 corner로. 순수 급변은 노이즈에 취약해
guard_deg(dorsum에서 충분히 꺾인 뒤) + presmooth 필요 → 결과는 V6에 가까움.

`max_left_deg`(30~45°)로 tip normal이 좌상단으로 그 각도를 넘지 않게 상한.

In [ ]:
def precise_contour_v7(mask, n=25, facing_thresh=2.5, smooth_win=3, clip_root=True, clip_drop_frac=1.0,
                       dthresh_deg=45.0, angle_win=3, presmooth=7, guard_deg=40.0, max_left_deg=40.0):
    tongue = (mask == LBL_TONGUE)
    if tongue.sum() < 10: return None
    cs = find_contours(tongue.astype(float), 0.5)
    if not cs: return None
    c = max(cs, key=len); airway = (mask == LBL_AIRWAY)
    if airway.sum() > 0:
        dt = distance_transform_edt(~airway)
        rr = np.clip(c[:,0].round().astype(int),0,mask.shape[0]-1); cc = np.clip(c[:,1].round().astype(int),0,mask.shape[1]-1)
        facing = dt[rr,cc] <= facing_thresh
        if facing.sum() >= 5:
            s, e = longest_true_run_cyclic(facing); c = c[np.arange(s,e)%len(c)]
    if c[0,1] > c[-1,1]: c = c[::-1]
    if clip_root: c = clip_posterior_spur(c, drop_frac=clip_drop_frac)   # root = V1/V5
    csm = c
    if presmooth > 1 and len(c) >= presmooth:                            # 계단 노이즈 완화
        k = np.ones(presmooth)/presmooth
        csm = np.column_stack([np.convolve(c[:,0],k,"same"), np.convolve(c[:,1],k,"same")])
    nv = outward_normals(csm, tongue)
    ang = np.unwrap(np.arctan2(nv[:,0], nv[:,1])); N = len(c)
    dang = np.array([np.degrees(abs(ang[min(N-1,i+angle_win)]-ang[max(0,i-angle_win)])) for i in range(N)])
    lang = np.degrees(np.arctan2(-nv[:,1], -nv[:,0]))                    # 좌측각(up=0, upper-left45=45)
    pk = int(np.argmin(c[:,0])); ad = ang[pk]; tip_idx = 0
    for i in range(pk, -1, -1):                                          # dorsum -> anterior
        if np.degrees(abs(ang[i]-ad)) > guard_deg and dang[i] > dthresh_deg:  # 충분히 꺾인 뒤 급변
            tip_idx = i; break
    cap_idx = 0                                                          # tip normal 좌측각 <= max_left_deg
    for i in range(pk, -1, -1):
        if lang[i] > max_left_deg: cap_idx = i+1; break
    tip_idx = max(tip_idx, cap_idx)                                      # 좌측 과연장 방지
    c = c[tip_idx:]
    return resample_rowcol(c, n, smooth_win)

show_row(precise_contour_v7, "V7")


## V8 — tip·root 둘 다 normal 기반
V7 tip(좌측각 cap) + **root도 normal**: dorsum에서 뒤로 걸으며 normal이 완전 우측(90°) 보기 직전(우측각 > right_thresh)을 root로.

In [ ]:
def precise_contour_v8(mask, n=25, facing_thresh=2.5, smooth_win=3, clip_root=True, clip_drop_frac=1.0,
                       dthresh_deg=45.0, angle_win=3, presmooth=7, guard_deg=40.0, max_left_deg=40.0, right_thresh=80.0):
    tongue = (mask == LBL_TONGUE)
    if tongue.sum() < 10: return None
    cs = find_contours(tongue.astype(float), 0.5)
    if not cs: return None
    c = max(cs, key=len); airway = (mask == LBL_AIRWAY)
    if airway.sum() > 0:
        dt = distance_transform_edt(~airway)
        rr = np.clip(c[:,0].round().astype(int),0,mask.shape[0]-1); cc = np.clip(c[:,1].round().astype(int),0,mask.shape[1]-1)
        facing = dt[rr,cc] <= facing_thresh
        if facing.sum() >= 5:
            s, e = longest_true_run_cyclic(facing); c = c[np.arange(s,e)%len(c)]
    if c[0,1] > c[-1,1]: c = c[::-1]
    csm = c
    if presmooth > 1 and len(c) >= presmooth:
        k = np.ones(presmooth)/presmooth; csm = np.column_stack([np.convolve(c[:,0],k,"same"), np.convolve(c[:,1],k,"same")])
    nv = outward_normals(csm, tongue); ang = np.unwrap(np.arctan2(nv[:,0], nv[:,1])); N = len(c)
    dang = np.array([np.degrees(abs(ang[min(N-1,i+angle_win)]-ang[max(0,i-angle_win)])) for i in range(N)])
    lang = np.degrees(np.arctan2(-nv[:,1], -nv[:,0])); rang = np.degrees(np.arctan2(nv[:,1], -nv[:,0]))
    pk = int(np.argmin(c[:,0])); ad = ang[pk]; tip_idx = 0
    for i in range(pk, -1, -1):
        if np.degrees(abs(ang[i]-ad)) > guard_deg and dang[i] > dthresh_deg: tip_idx = i; break
    cap_idx = 0
    for i in range(pk, -1, -1):
        if lang[i] > max_left_deg: cap_idx = i+1; break
    tip_idx = max(tip_idx, cap_idx)
    root_idx = N-1
    if clip_root:
        for i in range(pk, N):
            if rang[i] > right_thresh: root_idx = i-1; break
    root_idx = max(root_idx, pk+1)
    c = c[tip_idx:root_idx+1]
    return resample_rowcol(c, n, smooth_win)

show_row(precise_contour_v8, "V8")


## 모든 버전 normal vector 표시
V1~V5 각 contour 점의 **outward normal**(혀 밖=airway 쪽)을 빨간 화살표로. star=tip, square=root.
※ 위의 V1~V5 셀을 모두 먼저 실행해야 함(각 함수 사용).

In [ ]:
VERSIONS = [("V1", precise_contour_v1), ("V2", precise_contour_v2), ("V3", precise_contour_v3),
            ("V4", precise_contour_v4), ("V5", precise_contour_v5), ("V6", precise_contour_v6), ("V7", precise_contour_v7), ("V8", precise_contour_v8)]
L = 7.0   # 화살표 길이(px)
fig, ax = plt.subplots(len(VERSIONS), len(FRAMES), figsize=(2.9*len(FRAMES), 3.0*len(VERSIONS)))
for r, (name, fn) in enumerate(VERSIONS):
    for c, fr in enumerate(FRAMES):
        m = load2d(fr); ys, xs = np.where(m == LBL_TONGUE); a = np.atleast_2d(ax)[r, c]
        rc = fn(m, n=N_MARKERS)
        a.imshow(m, cmap="gray"); a.contour(m == LBL_AIRWAY, levels=[0.5], colors="deepskyblue", linewidths=0.5)
        if rc is not None:
            nv = outward_normals(rc, m == LBL_TONGUE)
            a.plot(rc[:,1], rc[:,0], "-", c="orange", lw=0.9)
            a.scatter(rc[:,1], rc[:,0], c=np.arange(len(rc)), cmap="viridis", s=10, zorder=4)
            a.quiver(rc[:,1], rc[:,0], nv[:,1]*L, nv[:,0]*L, color="red", angles="xy",
                     scale_units="xy", scale=1, width=0.007, zorder=5)
            a.scatter(rc[0,1], rc[0,0], marker="*", c="yellow", s=120, edgecolor="k", zorder=6)
            a.scatter(rc[-1,1], rc[-1,0], marker="s", c="cyan", s=45, edgecolor="k", zorder=6)
        a.axis("off"); a.set_xlim(xs.min()-12, xs.max()+12); a.set_ylim(ys.max()+12, ys.min()-14)
        if r == 0: a.set_title(f"f{fr}", fontsize=10)
    np.atleast_2d(ax)[r,0].text(-0.08, 0.5, name, transform=np.atleast_2d(ax)[r,0].transAxes,
                                rotation=90, va="center", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()
